Сейчас начнутся танцы с бубном (скорее всего)

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00


In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [3]:
df = pd.read_pickle('/content/dataset_after_eda.pkl')
df = df.drop_duplicates()
df.shape

(958, 211)

In [4]:
y = df['SI_above_median'].astype(int)
X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape[0])
print(X_test.shape[0])

766
192


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("regressor", LogisticRegression(max_iter=1000, random_state=42))]),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
}

baseline_results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    baseline_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="ROC-AUC", ascending=False)
df_baseline

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,CatBoost,0.6510,0.6275,0.6882,0.6564,0.7254
2,Gradient Boosting,0.6510,0.6300,0.6774,0.6528,0.7155
1,Random Forest,0.6771,0.6566,0.6989,0.6771,0.7002
0,Logistic Regression,0.6667,0.6436,0.6989,0.6701,0.6827


Все четыре модели показали схожий результат на параметрах по умолчанию

In [12]:
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5]
}

gb_param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5]
}

cat_param_grid = {
    'iterations': [300, 500],
    'learning_rate': [0.03, 0.05],
    'depth': [4, 6]
}

cv_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                     rf_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

cv_gb = GridSearchCV(GradientBoostingClassifier(random_state=42),
                     gb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

cv_cat = GridSearchCV(CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False),
                      cat_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

print("Random Forest")
cv_rf.fit(X_train, y_train)

print("Gradient Boosting")
cv_gb.fit(X_train, y_train)

print("CatBoost")
cv_cat.fit(X_train, y_train)

best_models = {
    "Random Forest": cv_rf.best_estimator_,
    "Gradient Boosting": cv_gb.best_estimator_,
    "CatBoost": cv_cat.best_estimator_
}

si_final_results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    si_final_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4)
    })

    print("Матрица ошибок для",name)
    print(confusion_matrix(y_test, y_pred))

df_si_final = pd.DataFrame(si_final_results).sort_values(by="ROC-AUC", ascending=False)
df_si_final

Random Forest
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Gradient Boosting
Fitting 3 folds for each of 18 candidates, totalling 54 fits
CatBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Матрица ошибок для Random Forest
[[65 34]
 [27 66]]
Матрица ошибок для Gradient Boosting
[[62 37]
 [30 63]]
Матрица ошибок для CatBoost
[[69 30]
 [29 64]]


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
2,CatBoost,0.6927,0.6809,0.6882,0.6845,0.7411
0,Random Forest,0.6823,0.6600,0.7097,0.6839,0.7138
1,Gradient Boosting,0.6510,0.6300,0.6774,0.6528,0.6789


GB после подбора параметров показал результат хуже чем до настройки, сетка не подошла. Для улучшения стоит попробовать меньшую глубину деревьев и более низкий learning rate

#Вывод
CatBoost опять лидирует и выбирается как финальная, результат устойчивый. Подбор параметров помог ему поднять ROC AUC с 0.7254 до 0.7411, а общую точность accuracy с 65.10% до 69.27%.
Random Forest сместился на второе место, но показал самый высокий recall, Полнота большая, значит что rf нашел почти 71% всех соединений с SI выше медианного значения.